In [1]:
# SAAM Project 2026 - Cell 0
# This cell should be run first.
# It loads the libraries, sets the main paths and parameters,
# and defines the helper functions used later in the notebook.

import re
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp

# =============================================================================
# Paths
# =============================================================================
# Use the current notebook folder as the base directory.

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "raw_data"
CLEAN_DIR = BASE_DIR / "cleaned_data"
OUT_DIR = BASE_DIR

# Create cleaned_data folder if it does not already exist
CLEAN_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Project settings
# =============================================================================

REGIONS = ["AMER", "EUR"]

START_YEAR = 2013
END_YEAR = 2024
WINDOW_YEARS = 10
MIN_VALID_MONTHS = 36
STALE_THRESHOLD = 0.50

# Raw files
STATIC_FILE = "Static_2025.csv"
RI_FILE = "DS_RI_T_USD_M_2025.csv"
MV_FILE = "DS_MV_T_USD_M_2025.csv"
CO2_FILE = "DS_CO2_SCOPE_1_Y_2025.csv"
REV_FILE = "DS_REV_Y_2025.csv"

# Cleaned/output files
RI_PRICES_FILE = "cleaned_RI_monthly_prices.csv"
RI_RETURNS_FILE = "cleaned_RI_monthly_returns.csv"
MV_CLEAN_FILE = "cleaned_MV_monthly.csv"
CO2_CLEAN_FILE = "cleaned_CO2_scope1_yearly.csv"
REV_CLEAN_FILE = "cleaned_REV_yearly.csv"
INV_SET_FILE = "investment_set_by_year.csv"
MVP_W_FILE = "mvp_weights_by_year.csv"
MVP_RET_FILE = "mvp_monthly_returns.csv"
VW_RET_FILE = "vw_monthly_returns.csv"


# =============================================================================
# Helper functions
# =============================================================================

def clean_headers(df):
    # Remove extra spaces from column names
    df.columns = [str(c).strip() for c in df.columns]
    return df


def get_id_cols(df):
    # Find the NAME and ISIN columns
    upper = {c.upper(): c for c in df.columns}
    return upper.get("NAME"), upper.get("ISIN")


def get_time_cols(df):
    # Return all columns except NAME and ISIN
    name_col, isin_col = get_id_cols(df)
    exclude = {c for c in [name_col, isin_col] if c is not None}
    return [c for c in df.columns if c not in exclude]


def sort_month_cols(cols):
    # Sort monthly columns in time order
    try:
        dt = pd.to_datetime(cols, errors="coerce")
        if dt.notna().all():
            return [c for _, c in sorted(zip(dt, cols))]
        return sorted(cols)
    except Exception:
        return sorted(cols)


def sort_year_cols(cols):
    # Sort yearly columns in time order
    def parse_year(x):
        try:
            return int(str(x)[:4])
        except Exception:
            return 999999
    return sorted(cols, key=parse_year)


def get_december_col(month_cols, year):
    # Return the December column for a given year
    dts = pd.to_datetime(month_cols, errors="coerce")
    candidates = [c for c, d in zip(month_cols, dts)
                  if pd.notna(d) and d.year == year and d.month == 12]
    return candidates[-1] if candidates else None


def get_trailing_window_cols(month_cols, end_year, window_years=10):
    # Return the monthly columns in the rolling estimation window
    # Example: if end_year = 2013, this gives Jan 2004 to Dec 2013
    end_dt = pd.Timestamp(f"{end_year}-12-31")
    start_dt = end_dt - pd.DateOffset(years=window_years)
    col_dates = pd.to_datetime(month_cols, errors="coerce")

    return [c for c, d in zip(month_cols, col_dates)
            if pd.notna(d) and start_dt < d <= end_dt]


def get_investment_year_cols(month_cols, investment_year):
    # Return the monthly columns in the investment year
    col_dates = pd.to_datetime(month_cols, errors="coerce")
    return [c for c, d in zip(month_cols, col_dates)
            if pd.notna(d) and d.year == investment_year]


def drop_empty_timeseries_rows(df):
    # Remove firms where all time-series values are missing
    df = clean_headers(df)
    _, isin_col = get_id_cols(df)
    time_cols = get_time_cols(df)

    df = df.dropna(subset=[isin_col]).copy()
    df[time_cols] = df[time_cols].apply(pd.to_numeric, errors="coerce")

    return df.loc[~df[time_cols].isna().all(axis=1)].copy()


def keep_isins(df, valid_isins):
    # Keep only firms in the chosen ISIN set
    _, isin_col = get_id_cols(df)
    return df[df[isin_col].isin(valid_isins)].copy()


def annual_forward_fill(df):
    # Forward-fill annual data using the previous available year.
    # This is used for CO2 and revenue.
    out = df.copy()
    year_cols = sort_year_cols(get_time_cols(out))
    out[year_cols] = out[year_cols].apply(pd.to_numeric, errors="coerce")
    out[year_cols] = out[year_cols].ffill(axis=1)
    return out


def detect_delisting(df):
    # Some companies include delisting information in the NAME column.
    # If we find a date in the name, we assume that is the delisting date
    # and set the price to 0 from that point onward.
    #
    # If the name only says "DEAD", we set the first month after the last
    # observed price to 0.
    #
    # This ensures we correctly capture the -100% return when a firm delists.

    out = df.copy()
    name_col, _ = get_id_cols(out)

    if name_col is None:
        return out

    month_cols = sort_month_cols(get_time_cols(out))
    month_dates = pd.to_datetime(month_cols, errors="coerce")

    # Look for a date in the format DD/MM/YYYY inside the name
    date_pattern = re.compile(r"\d{2}/\d{2}/\d{4}")

    for idx in out.index:
        name = str(out.at[idx, name_col]).upper()

        # Case 1: name contains both a date and the word "DELIST"
        if "DELIST" in name:
            match = date_pattern.search(name)

            if match:
                try:
                    delist_date = pd.to_datetime(match.group(), dayfirst=True)
                except:
                    continue

                for col, dt in zip(month_cols, month_dates):
                    if pd.notna(dt) and dt >= delist_date:
                        out.at[idx, col] = 0.0

        # Case 2: name contains "DEAD" but no date
        elif "DEAD" in name:
            values = out.loc[idx, month_cols]
            last_valid = values.last_valid_index()

            if last_valid is not None and last_valid != month_cols[-1]:
                next_pos = month_cols.index(last_valid) + 1
                out.at[idx, month_cols[next_pos]] = 0.0

    return out

def compute_monthly_returns(ri_prices):
    # Compute simple monthly returns from cleaned RI prices.
    # Standard: R_t = P_t / P_{t-1} - 1
    # Delisting: R = -1.0 (price goes to 0)
    # Gap: R = NaN

    df = ri_prices.copy()
    name_col, isin_col = get_id_cols(df)
    month_cols = sort_month_cols(get_time_cols(df))
    id_cols_list = [c for c in [name_col, isin_col] if c is not None]

    returns = df[id_cols_list].copy()

    for i, col in enumerate(month_cols):
        if i == 0:
            returns[col] = np.nan
            continue

        prev_col = month_cols[i - 1]
        prev = df[prev_col].values.astype(float)
        curr = df[col].values.astype(float)
        ret = np.full(len(df), np.nan)

        valid = (~np.isnan(prev)) & (prev > 0) & (~np.isnan(curr)) & (curr > 0)
        ret[valid] = curr[valid] / prev[valid] - 1.0

        delist = (curr == 0.0) & (~np.isnan(prev)) & (prev > 0)
        ret[delist] = -1.0

        returns[col] = ret

    return returns


def flag_stale_prices(returns_df, end_year, window_years=10, threshold=0.50):
    # Return a DataFrame with stale-price diagnostics per firm over the
    # trailing window_years-year window ending December of end_year.

    _, isin_col = get_id_cols(returns_df)
    month_cols = sort_month_cols(get_time_cols(returns_df))
    window_cols = get_trailing_window_cols(
        month_cols, end_year=end_year, window_years=window_years
    )

    # If no valid trailing window is found, return an empty diagnostics table
    if not window_cols:
        return pd.DataFrame(columns=["ISIN", "n_valid", "n_zero", "zero_frac", "stale_flag"])

    # Keep only the returns in the estimation window
    data = returns_df[window_cols].apply(pd.to_numeric, errors="coerce")

    # Count available observations and zero returns for each firm
    n_valid = data.notna().sum(axis=1)
    n_zero = (data == 0.0).sum(axis=1)
    zero_frac = n_zero / n_valid.replace(0, np.nan)

    out = pd.DataFrame({
        "ISIN": returns_df[isin_col].values,
        "n_valid": n_valid.values,
        "n_zero": n_zero.values,
        "zero_frac": zero_frac.values,
    })

    # Flag firms with too many zero returns as stale-price firms
    out["stale_flag"] = out["zero_frac"] > threshold

    return out


def estimate_moments(ri_ret_by_isin, isins, window_cols):
    # Estimate mean returns and covariance matrix over the estimation window.
    # Only keep firms with no missing data.

    ret_mat = (
        ri_ret_by_isin.reindex(isins)[window_cols]
        .apply(pd.to_numeric, errors="coerce")
    )

    # Keep only firms with full data
    complete_mask = ret_mat.notna().all(axis=1)
    ret_complete = ret_mat.loc[complete_mask]
    valid_isins = ret_complete.index.tolist()

    if not valid_isins:
        return None, None, []

    # Convert to matrix (N x tau)
    R = ret_complete.to_numpy(dtype=float)
    tau = float(R.shape[1])

    # Mean returns
    mu = R.mean(axis=1)

    # Covariance matrix
    R_c = R - mu[:, np.newaxis]
    cov = (R_c @ R_c.T) / tau

    # Ensure covariance is well-behaved
    cov = 0.5 * (cov + cov.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    cov = eigvecs @ np.diag(np.clip(eigvals, 1e-8, None)) @ eigvecs.T
    cov = 0.5 * (cov + cov.T)

    return mu, cov, valid_isins


def solve_mvp(cov):
    # Solve long-only minimum variance portfolio

    n = cov.shape[0]
    w = cp.Variable(n)

    prob = cp.Problem(
        cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov))),
        [cp.sum(w) == 1, w >= 0]
    )

    prob.solve(solver=cp.OSQP, verbose=False)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"MVP optimisation failed: {prob.status}")

    weights = np.clip(np.array(w.value).flatten(), 0, None)

    return weights / weights.sum()


def summary_stats(series, name):
    ann = 12
    vals = series.to_numpy(dtype=float)
    mu = vals.mean()
    std = vals.std(ddof=1)

    return {
        "portfolio": name,
        "start_date": series.index.min().strftime("%Y-%m"),
        "end_date": series.index.max().strftime("%Y-%m"),
        "n_months": len(vals),
        "ann_mean_return": mu * ann,
        "ann_volatility": std * np.sqrt(ann),
        "sharpe_ratio": np.nan if std == 0 else (mu / std) * np.sqrt(ann),
        "min_monthly_return": vals.min(),
        "max_monthly_return": vals.max(),
        "variance":          float(port_variance),
    }


print("Cell 0 complete: helpers and config loaded.")


Cell 0 complete: helpers and config loaded.


## Section 1 — Data Cleaning

Raw Datastream files are loaded and cleaned in five steps:
1. Filter to the assigned region universe (North America + Europe)
2. Detect and handle delisted firms (price set to 0, return = -100%)
3. Treat RI prices below 0.5 as missing to avoid infinite returns
4. Forward-fill missing annual CO2 and revenue data (middle/end gaps only)
5. Intersect a common ISIN universe across all files

Note: stale-price screening is **not** applied here — it is applied year-by-year in Cell 2 to avoid any look-ahead bias.

In [2]:
####################################################################################
# SAAM Project 2026 - Part I
# CELL 1 - DATA CLEANING
####################################################################################

# 1. LOAD RAW FILES
static = clean_headers(pd.read_csv(RAW_DIR / STATIC_FILE))
ri     = clean_headers(pd.read_csv(RAW_DIR / RI_FILE))
mv     = clean_headers(pd.read_csv(RAW_DIR / MV_FILE))
co2    = clean_headers(pd.read_csv(RAW_DIR / CO2_FILE))
rev    = clean_headers(pd.read_csv(RAW_DIR / REV_FILE))

# 2. FILTER STATIC TO ASSIGNED REGIONS
static_name_col, static_isin_col = get_id_cols(static)
if static_isin_col is None:
    raise ValueError("Static file must contain an ISIN column.")
if "Region" not in static.columns:
    raise ValueError("Static file must contain a 'Region' column.")

static = static.dropna(subset=[static_isin_col]).copy()
static = static[static["Region"].isin(REGIONS)].copy()

# 3. DROP ROWS WITH NO TIME-SERIES DATA AT ALL
ri  = drop_empty_timeseries_rows(ri)
mv  = drop_empty_timeseries_rows(mv)
co2 = drop_empty_timeseries_rows(co2)
rev = drop_empty_timeseries_rows(rev)

# 4. RESTRICT TO REGION UNIVERSE
region_isins = set(static[static_isin_col])
ri  = keep_isins(ri,  region_isins)
mv  = keep_isins(mv,  region_isins)
co2 = keep_isins(co2, region_isins)
rev = keep_isins(rev, region_isins)

# 5. CLEAN RI: delisting -> low prices -> compute returns
ri_month_cols = sort_month_cols(get_time_cols(ri))
ri[ri_month_cols] = ri[ri_month_cols].apply(pd.to_numeric, errors="coerce")

# Delisting must be flagged BEFORE the low-price filter (delist sets price to exact 0)
ri = detect_delisting(ri)

# Treat prices in (0, 0.5) as missing; preserve exact 0.0 for delisted firms
for c in ri_month_cols:
    mask = (ri[c] < 0.5) & (ri[c] > 0)
    ri.loc[mask, c] = np.nan

ri     = ri.loc[~ri[ri_month_cols].isna().all(axis=1)].copy()
ri_ret = compute_monthly_returns(ri)

# 6. CLEAN MV (numeric conversion only; no fill needed for monthly data)
mv_month_cols = sort_month_cols(get_time_cols(mv))
mv[mv_month_cols] = mv[mv_month_cols].apply(pd.to_numeric, errors="coerce")

# 7. FORWARD-FILL ANNUAL DATA (middle/end gaps only; beginning gaps stay NaN)
co2 = annual_forward_fill(co2)
rev = annual_forward_fill(rev)

# 8. INTERSECT COMMON ISIN UNIVERSE ACROSS ALL FILES
_, ri_isin_col  = get_id_cols(ri)
_, mv_isin_col  = get_id_cols(mv)
_, co2_isin_col = get_id_cols(co2)
_, rev_isin_col = get_id_cols(rev)

common_isins = (
    set(static[static_isin_col])
    & set(ri[ri_isin_col])
    & set(mv[mv_isin_col])
    & set(co2[co2_isin_col])
    & set(rev[rev_isin_col])
)
# NOTE: stale-price screening is NOT applied globally here.
# It must be done year-by-year in Cell 2 to avoid look-ahead bias.

static = keep_isins(static, common_isins).sort_values(static_isin_col).reset_index(drop=True)
ri     = keep_isins(ri,     common_isins).sort_values(ri_isin_col).reset_index(drop=True)
ri_ret = keep_isins(ri_ret, common_isins).sort_values(ri_isin_col).reset_index(drop=True)
mv     = keep_isins(mv,     common_isins).sort_values(mv_isin_col).reset_index(drop=True)
co2    = keep_isins(co2,    common_isins).sort_values(co2_isin_col).reset_index(drop=True)
rev    = keep_isins(rev,    common_isins).sort_values(rev_isin_col).reset_index(drop=True)

# 9. SAVE
static.to_csv(CLEAN_DIR / "cleaned_static.csv", index=False)
ri.to_csv(    CLEAN_DIR / RI_PRICES_FILE,        index=False)
ri_ret.to_csv(CLEAN_DIR / RI_RETURNS_FILE,       index=False)
mv.to_csv(    CLEAN_DIR / MV_CLEAN_FILE,         index=False)
co2.to_csv(   CLEAN_DIR / CO2_CLEAN_FILE,        index=False)
rev.to_csv(   CLEAN_DIR / REV_CLEAN_FILE,        index=False)

print("Cell 1 - Data cleaning complete.")
print(f"  Common universe: {len(common_isins)} firms")
print(f"  Saved to: {CLEAN_DIR}")

/var/folders/3j/295v0dr15nl_1gcd4nxlgj240000gn/T/ipykernel_27483/3728783153.py:234: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  returns[col] = ret
/var/folders/3j/295v0dr15nl_1gcd4nxlgj240000gn/T/ipykernel_27483/3728783153.py:234: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  returns[col] = ret
/var/folders/3j/295v0dr15nl_1gcd4nxlgj240000gn/T/ipykernel_27483/3728783153.py:234: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

Cell 1 - Data cleaning complete.
  Common universe: 1209 firms
  Saved to: /Users/saarj/Documents/The Drive/UNIL Masters/Year 1/Spring Semester/Sustainability Aware Asset Management/AS- Sustainability Aware Asset Management Project /cleaned_data


# Section 2.1 - Investment Set

## Objective
For each screen year Y = 2013 to 2024, define the list of firms eligible
for investment in year Y+1.

---

## Step 1: Apply the three inclusion criteria

At the end of year Y, include a firm if and only if it satisfies all three:

**Criterion 1 - Region**
The firm belongs to your assigned region.

**Criterion 2 - Sufficient return history**
Within the 10-year trailing window (Jan of Y-9 to Dec of Y), the firm has
at least 36 months of non-missing returns.
- omega = 120 months total in the window
- Minimum required = 36 months (3 years)
- If valid months < 36, exclude the firm for year Y+1

**Criterion 3 - Carbon data availability**
The firm has carbon data (Scope 1 or Scope 2 emissions and revenues)
available at the end of year Y.
- Forward-filling was already done in Section 1 (data cleaning)
- Here you only check: does a valid carbon value exist at end of year Y?
- If not, exclude the firm for year Y+1

All three criteria must be satisfied simultaneously.

---

## Step 2: Estimate mu and Sigma for each passing firm

Using only the valid monthly returns within the 10-year trailing window:

**Expected return (biased estimator):**

$$\hat{\mu}_Y = \frac{1}{\omega} \sum_{k=0}^{\omega-1} R_{t-k}$$

**Covariance matrix (biased estimator):**

$$\hat{\Sigma}_Y = \frac{1}{\omega} \sum_{k=0}^{\omega-1}
(R_{t-k} - \hat{\mu}_Y)(R_{t-k} - \hat{\mu}_Y)^\top$$

where omega = 120 (division by omega, not omega - 1).

---

## Step 3: Output the investment set table

Produce one row per ISIN per screen year with an investable flag.

| Column                   | Description                                  |
|--------------------------|----------------------------------------------|
| screen_year              | Year Y at which the decision is made         |
| investment_year          | Year Y+1 in which the portfolio is held      |
| ISIN                     | Firm identifier                              |
| investable_for_next_year | True / False                                 |

This table is the direct input to Section 2.2.


In [ ]:
####################################################################################
# SAAM Project 2026 - Part I
# CELL 2 - SECTION 2.1: INVESTMENT SET
####################################################################################

####################################################################################
# SECTION 2.1 - STEP 1, CRITERION 1: Region filter
####################################################################################

# Load the static file (ISIN, Name, Country, Region)
static = clean_headers(pd.read_csv(RAW_DIR / STATIC_FILE))

# Keep only firms belonging to the assigned regions
region_col = [c for c in static.columns if c.upper() == "REGION"][0]
isin_col   = [c for c in static.columns if c.upper() == "ISIN"][0]

region_mask    = static[region_col].isin(REGIONS)
eligible_isins = static.loc[region_mask, isin_col].unique().tolist()

print(f"Total firms in static file : {len(static)}")
print(f"Firms after region filter  : {len(eligible_isins)}")
print(f"Regions kept               : {REGIONS}")

####################################################################################
# SECTION 2.1 - STEP 1, CRITERION 2: Sufficient return history
####################################################################################

# Load the cleaned monthly returns
ri_returns = clean_headers(pd.read_csv(CLEAN_DIR / RI_RETURNS_FILE))

# Find the ISIN column and isolate the date columns
isin_col      = [c for c in ri_returns.columns if c.upper() == "ISIN"][0]
all_date_cols = [c for c in ri_returns.columns if c != isin_col]

# Work only on the firms that passed Criterion 1
ri_returns = ri_returns[ri_returns[isin_col].isin(eligible_isins)]
ri_returns = ri_returns.set_index(isin_col)

# Convert all date columns to datetime so we can filter by year easily
date_index = pd.to_datetime(all_date_cols, errors="coerce")

criterion2_results = []

for year in range(START_YEAR, END_YEAR + 1):

    # The trailing window is the 10 years of monthly data ending in December of year Y.
    # Example: for Y=2013, we look at Jan 2004 to Dec 2013 (120 months).
    window_end   = pd.Timestamp(f"{year}-12-31")
    window_start = window_end - pd.DateOffset(years=WINDOW_YEARS)

    # Pick the date columns that fall strictly inside the window
    # (window_start, window_end] means we exclude the start boundary but include the end
    window_cols = [
        c for c, d in zip(all_date_cols, date_index)
        if pd.notna(d) and window_start < d <= window_end
    ]

    # Extract the return values for the window: shape (N_firms, N_months)
    window_returns = ri_returns[window_cols].apply(pd.to_numeric, errors="coerce")

    # Count how many months each firm has a non-NaN return in the window.
    # A NaN means the firm had no valid price that month (cleaned in Section 1).
    valid_months = window_returns.notna().sum(axis=1)

    # A firm passes if it has at least MIN_VALID_MONTHS (36) valid observations.
    # 36 months = 3 years, which is the minimum to get a meaningful covariance estimate.
    passes = valid_months >= MIN_VALID_MONTHS

    for isin in ri_returns.index:
        criterion2_results.append({
            "screen_year":       year,
            "ISIN":              isin,
            "valid_months":      int(valid_months[isin]),
            "passes_criterion2": bool(passes[isin]),
        })

criterion2_df = pd.DataFrame(criterion2_results)

# Quick sanity check per year
summary = (criterion2_df
           .groupby("screen_year")["passes_criterion2"]
           .sum()
           .rename("firms_passing_criterion2"))
print(summary.to_string())

####################################################################################
# SECTION 2.1 - STEP 1, CRITERION 3: Carbon data availability
####################################################################################

# Load cleaned CO2 and revenue data 
co2_clean = clean_headers(pd.read_csv(CLEAN_DIR / CO2_CLEAN_FILE))
rev_clean = clean_headers(pd.read_csv(CLEAN_DIR / REV_CLEAN_FILE))

co2_clean.columns = [str(c) for c in co2_clean.columns]
rev_clean.columns = [str(c) for c in rev_clean.columns]

isin_col_co2 = [c for c in co2_clean.columns if c.upper() == "ISIN"][0]
isin_col_rev = [c for c in rev_clean.columns if c.upper() == "ISIN"][0]

co2_clean = co2_clean.set_index(isin_col_co2)
rev_clean = rev_clean.set_index(isin_col_rev)

criterion3_results = []

for year in range(START_YEAR, END_YEAR + 1):
    # Carbon data is annual, so we check if the column for year Y exists and is non-NaN.
    # Both CO2 and revenue must be available to compute carbon intensity in Part II.
    year_str = str(year)

    co2_available = co2_clean[year_str].notna() if year_str in co2_clean.columns else pd.Series(False, index=co2_clean.index)
    rev_available = rev_clean[year_str].notna() if year_str in rev_clean.columns else pd.Series(False, index=rev_clean.index)

    passes = co2_available & rev_available

    for isin in co2_clean.index:
        criterion3_results.append({
            "screen_year":       year,
            "ISIN":              isin,
            "passes_criterion3": bool(passes.get(isin, False)),
        })

criterion3_df = pd.DataFrame(criterion3_results)

summary = (criterion3_df
           .groupby("screen_year")["passes_criterion3"]
           .sum()
           .rename("firms_passing_criterion3"))
print(summary.to_string())


####################################################################################
# SECTION 2.1 - STEP 2: Estimate mu and Sigma for each passing firm
####################################################################################

# Merge all three criteria into one master table
inv_set = (criterion2_df
           .merge(criterion3_df, on=["screen_year", "ISIN"])
           .merge(pd.DataFrame({"ISIN": eligible_isins}), on="ISIN"))

# investable = passed region filter (C1) AND sufficient history (C2) AND carbon data (C3)
inv_set["investable"] = (
    inv_set["ISIN"].isin(eligible_isins) &
    inv_set["passes_criterion2"] &
    inv_set["passes_criterion3"]
)

moments_by_year = {}  # stores {"year": {"mu": ..., "cov": ..., "isins": ...}}

for year in range(START_YEAR, END_YEAR + 1):

    # ISINs that passed all criteria for this screen year
    mask   = (inv_set["screen_year"] == year) & inv_set["investable"]
    isins  = inv_set.loc[mask, "ISIN"].tolist()

    # Same trailing window as Criterion 2
    window_end   = pd.Timestamp(f"{year}-12-31")
    window_start = window_end - pd.DateOffset(years=WINDOW_YEARS)
    window_cols  = [
        c for c, d in zip(all_date_cols, date_index)
        if pd.notna(d) and window_start < d <= window_end
    ]

    # Extract returns for investable firms, keep only fully complete rows (no NaNs).
    # We need a complete matrix to compute Sigma without missing entries.
    ret = (ri_returns
           .reindex(isins)[window_cols]
           .apply(pd.to_numeric, errors="coerce"))
    ret = ret.dropna()  # drops firms with any NaN in the window

    if ret.empty:
        print(f"{year}: no complete firms, skipping.")
        continue

    # ret shape is (N_firms, ω=omega) so we transpose to (ω, N_firms) for the formulas
    R   = ret.values.T          # shape (ω, N)
    omega = R.shape[0]          # number of months in window (should be 120)

    # mu_Y = (1/ω) * sum of R_t-k  =>  identical to .mean(axis=0)
    mu  = R.mean(axis=0)        # shape (N,)

    # Sigma_Y = (1/ω) * sum of (R_t-k - mu)'(R_t-k - mu)
    # This is the BIASED estimator (divide by omega, not omega-1) as in the PDF
    centered = R - mu
    cov      = (centered.T @ centered) / omega   # shape (N, N)
    
    moments_by_year[year] = {"mu": mu, "cov": cov, "isins": ret.index.tolist()}

    print(f"{year}: {len(ret)} firms, omega={omega}")
    
####################################################################################
# SECTION 2.1 - STEP 3: Output the investment set table
####################################################################################
# Update investable flag: only keep firms that survived dropna() in step 2
# (firms with any NaN in the window were dropped from the covariance estimation)
for year, data in moments_by_year.items():
    complete_isins = data["isins"]
    inv_set.loc[
        (inv_set["screen_year"] == year) & ~inv_set["ISIN"].isin(complete_isins),
        "investable"
    ] = False

# Save
inv_set_output = inv_set[["screen_year", "ISIN", "investable"]].copy()
inv_set_output["investment_year"] = inv_set_output["screen_year"] + 1
inv_set_output.to_csv(CLEAN_DIR / INV_SET_FILE, index=False)

# Sanity check
summary = (inv_set_output
           .groupby("screen_year")["investable"]
           .sum()
           .rename("investable_firms"))
print(summary.to_string())

Total firms in static file : 2545
Firms after region filter  : 1302
Regions kept               : ['AMER', 'EUR']
screen_year
2013    1185
2014    1187
2015    1187
2016    1191
2017    1191
2018    1194
2019    1195
2020    1196
2021    1197
2022    1198
2023    1198
2024    1202
screen_year
2013     837
2014     863
2015     896
2016     918
2017     952
2018    1010
2019    1100
2020    1143
2021    1175
2022    1199
2023    1206
2024    1209
2013: 771 firms, omega=120
2014: 811 firms, omega=120
2015: 862 firms, omega=120
2016: 895 firms, omega=120
2017: 931 firms, omega=120
2018: 984 firms, omega=120
2019: 1073 firms, omega=120


/var/folders/3j/295v0dr15nl_1gcd4nxlgj240000gn/T/ipykernel_27483/1634916321.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_index = pd.to_datetime(all_date_cols, errors="coerce")


2020: 1115 firms, omega=120
2021: 1148 firms, omega=120
2022: 1147 firms, omega=120
2023: 1136 firms, omega=120
2024: 1120 firms, omega=120
screen_year
2013     771
2014     811
2015     862
2016     895
2017     931
2018     984
2019    1073
2020    1115
2021    1148
2022    1147
2023    1136
2024    1120


## 2.2 MVP

We determine the allocation at the end of each year Y for the next year Y+1. The weights have to add up to 1 and be bigger or equal to 0 when solving for the MVP Minimization problem:

$$\min_{\alpha_Y} \; \sigma_{p;Y}^2 = \alpha_Y' \Sigma_Y \alpha_Y \quad \text{s.t.} \quad \alpha_Y' e = 1, \quad \alpha_{i,Y} \geq 0$$

The number of firms that are missing any return values are not taken in calculation for the covariance matrix. Thus we avoid pairwise covariance and NaN-zeroing heuristics.

In [ ]:
####################################################################################
# SAAM Project 2026 - Part I
# CELL 3 - SECTION 2.2: MINIMUM VARIANCE PORTFOLIO WEIGHTS
####################################################################################

ri_returns      = clean_headers(pd.read_csv(CLEAN_DIR / RI_RETURNS_FILE))
_, ret_isin_col  = get_id_cols(ri_returns)
ret_month_cols   = sort_month_cols(get_time_cols(ri_returns))
ri_ret_by_isin   = ri_returns.set_index(ret_isin_col)

weights_rows      = []
year_summary_rows = []

for year in range(START_YEAR, END_YEAR + 1):
    allocation_year = year + 1

    investable_isins = investment_set.loc[
        (investment_set["screen_year"] == year)
        & investment_set["investable_for_next_year"],
        "ISIN"
    ].tolist()

    if not investable_isins:
        print(f"{year}is skipped. There are no investable firms.")
        continue

    window_cols = get_trailing_window_cols(ret_month_cols, end_year=year, window_years=WINDOW_YEARS)
    if not window_cols:
        print(f" {year} is skipped. There are no trailing window columns.")
        continue

    mu, cov, valid_isins = estimate_moments(ri_ret_by_isin, investable_isins, window_cols)

    if not valid_isins:
        print(f"{year} is skipped. There are no complete-case assets available.")
        continue

    weights = solve_mvp(cov)

    port_variance = weights.T @ cov @ weights

    for isin, w in zip(valid_isins, weights):
        weights_rows.append({
            "screen_year":     year,
            "investment_year": investment_year,
            "ISIN":            isin,
            "weight":          w,
        })

    year_summary_rows.append({
        "screen_year":       year,
        "investment_year":   investment_year,
        "n_investable":      len(investable_isins),
        "n_assets_in_cov":   len(valid_isins),
        "n_dropped_missing": len(investable_isins) - len(valid_isins),
        "max_weight":        float(weights.max()),
    })
    print(f" For {year} there are {len(investable_isins)} firms. Only {len(valid_isins)} were used. "
          f"The variance of the portfolio is {port_variance:.6f}. The maximum allocation was {weights.max():.4f}")

weights_by_year = pd.DataFrame(weights_rows)
weights_by_year.to_csv(OUT_DIR / MVP_W_FILE, index=False)
pd.DataFrame(year_summary_rows).to_csv(OUT_DIR / "mvp_weights_summary_by_year.csv", index=False)

NameError: name 'investment_set' is not defined

In [ ]:
####################################################################################
# SECTION 2.3 - VALUE-WEIGHTED BENCHMARK AND COMPARISON
####################################################################################

# This section builds the value-weighted benchmark portfolio and then compares
# it with the minimum variance portfolio.

# =============================================================================
# 2.3A Load monthly market cap data
# =============================================================================

mv_clean = clean_headers(pd.read_csv(CLEAN_DIR / MV_CLEAN_FILE))
_, mv_isin_col = get_id_cols(mv_clean)
mv_month_cols = sort_month_cols(get_time_cols(mv_clean))
mv_by_isin = mv_clean.set_index(mv_isin_col)

# =============================================================================
# 2.3B Build monthly value-weighted benchmark returns
# =============================================================================

vw_rows = []

for year in range(START_YEAR, END_YEAR + 1):
    investment_year = year + 1

    # Use the same investment universe as in Section 2.1
    investable_isins = investment_set.loc[
        (investment_set["screen_year"] == year)
        & (investment_set["investable_for_next_year"] == True),
        "ISIN"
    ].tolist()

    if not investable_isins:
        continue

    inv_cols = get_investment_year_cols(ret_month_cols, investment_year)
    if not inv_cols:
        continue

    for col in inv_cols:
        current_dt = pd.Timestamp(col)
        prev_month_dt = current_dt - pd.DateOffset(months=1)

        # Use market cap from the previous month to form value weights
        prev_candidates = [c for c in mv_month_cols if pd.Timestamp(c) == prev_month_dt]
        if not prev_candidates:
            continue

        prev_col = prev_candidates[0]

        caps = (
            mv_by_isin.reindex(investable_isins)[prev_col]
            .apply(pd.to_numeric, errors="coerce")
        )

        rets = (
            ri_ret_by_isin.reindex(investable_isins)[col]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0.0)
        )

        # Only keep firms with valid positive market cap
        valid = caps.notna() & (caps > 0)
        caps = caps[valid]
        rets = rets[valid]

        if caps.empty:
            continue

        weights_vw = caps / caps.sum()
        rp_vw = float((weights_vw * rets).sum())

        vw_rows.append({
            "screen_year": year,
            "investment_year": investment_year,
            "date": current_dt,
            "Rp_vw": rp_vw
        })

vw_monthly_returns = pd.DataFrame(vw_rows).sort_values("date").reset_index(drop=True)

if vw_monthly_returns.empty:
    raise RuntimeError("No value-weighted returns were generated.")

# =============================================================================
# 2.3C Merge MVP and benchmark returns
# =============================================================================

compare_df = (
    mvp_monthly_returns[["date", "Rp_mvp"]]
    .assign(date=lambda x: pd.to_datetime(x["date"]))
    .merge(
        vw_monthly_returns[["date", "Rp_vw"]]
        .assign(date=lambda x: pd.to_datetime(x["date"])),
        on="date",
        how="inner"
    )
    .sort_values("date")
    .set_index("date")
)

if compare_df.empty:
    raise RuntimeError("Merged comparison dataframe is empty.")

# Compute cumulative returns
compare_df["cum_mvp"] = (1.0 + compare_df["Rp_mvp"]).cumprod() - 1.0
compare_df["cum_vw"] = (1.0 + compare_df["Rp_vw"]).cumprod() - 1.0

# =============================================================================
# 2.3D Compare performance statistics
# =============================================================================

mvp_stats = summary_stats(compare_df["Rp_mvp"], "P(mv)_oos")
vw_stats = summary_stats(compare_df["Rp_vw"], "P(vw)_oos")

comparison_summary = pd.DataFrame([mvp_stats, vw_stats])

# =============================================================================
# 2.3E Table 1 - Performance summary
# =============================================================================

display_df = comparison_summary[
    ["portfolio", "ann_mean_return", "ann_volatility",
     "sharpe_ratio", "min_monthly_return", "max_monthly_return"]
].copy()

display_df.columns = [
    "Portfolio", "Ann. Return", "Ann. Volatility",
    "Sharpe Ratio", "Min Monthly Ret.", "Max Monthly Ret."
]

for col in ["Ann. Return", "Ann. Volatility", "Min Monthly Ret.", "Max Monthly Ret."]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.2%}")

display_df["Sharpe Ratio"] = display_df["Sharpe Ratio"].apply(lambda x: f"{x:.3f}")

print("\n" + "=" * 72)
print("Table 1 — Performance Summary: MVP vs Value-Weighted Benchmark")
print(f"          Period: {compare_df.index.min().strftime('%b %Y')} to {compare_df.index.max().strftime('%b %Y')}")
print("=" * 72)
print(display_df.to_string(index=False))
print("=" * 72)

# =============================================================================
# 2.3F Table 2 - Number of investable firms by year
# =============================================================================

inv_by_year = (
    investment_set[investment_set["investable_for_next_year"]]
    .groupby("screen_year")["ISIN"]
    .count()
    .rename("N Firms")
    .reset_index()
    .rename(columns={"screen_year": "Screen Year"})
)

print("\n" + "=" * 35)
print("Table 2 — Investable Universe by Year")
print("=" * 35)
print(inv_by_year.to_string(index=False))
print("=" * 35)

# =============================================================================
# 2.3G Table 3 - Largest MVP holdings in first portfolio
# =============================================================================

first_screen_year = weights_by_year["screen_year"].min()

top10 = (
    weights_by_year[weights_by_year["screen_year"] == first_screen_year]
    .nlargest(10, "weight")[["ISIN", "weight"]]
    .copy()
    .reset_index(drop=True)
)

top10.index = range(1, 11)
top10["weight"] = top10["weight"].apply(lambda x: f"{x:.2%}")

print("\n" + "=" * 45)
print(f"Table 3 — Top 10 MVP Holdings (portfolio for {first_screen_year + 1})")
print("=" * 45)
print(top10.to_string(columns=["ISIN", "weight"]))
print("=" * 45)

# =============================================================================
# 2.3H Save outputs
# =============================================================================

vw_monthly_returns.to_csv(OUT_DIR / VW_RET_FILE, index=False)
comparison_summary.to_csv(OUT_DIR / "mvp_vs_vw_performance_summary.csv", index=False)
compare_df.to_csv(OUT_DIR / "mvp_vs_vw_monthly_comparison.csv", index=True)
compare_df[["cum_mvp", "cum_vw"]].to_csv(OUT_DIR / "mvp_vs_vw_cumulative_returns.csv", index=True)

print("\nSection 2.3 complete.")